# Calibration X6Y3 — the qcal chain on the real chip (ZCU216)

The hardware notebook **`Calibration_X6Y3_single_QCal.ipynb`** (qcal + QubiC, X6Y3 session of
**2026-07-26**), reproduced step-for-step with `riscq.cal` on a **real ZCU216** driving the X6Y3
chip (spec [13-qcal-parity](../specs/software/13-qcal-parity.md), re-parity in
[20](../specs/software/20-single-qcal-notebook-parity.md)). The reference hops between
commented-out variants of each sweep; the table below states which one this notebook mirrors — the
invocations that session actually ran.

The config of record is the real X6Y3 **qcal tree** ([`cal-config-x6y3.yaml`](cal-config-x6y3.yaml)):
8 qubits, FAST_DRAG X90s, readout at 6.55–6.84 GHz, `readout/herald: true`, 500 µs passive reset —
loaded with `Config.from_qcal`, calibrated fields written back with `save_qcal`.

Like the other hardware notebooks (`remote_pulse`, `vna`, `iq_scatter`) this connects to the
board server over `RemoteDriver` and is **not executed in CI** — its co-sim twin
([`calibration_x6y3_cosim.ipynb`](calibration_x6y3_cosim.ipynb)) runs the same chain against a
planted `TwoLevelModel` and is the verified reference for every code path used here except
the two `Resonator` scans, which it does not run — that class is gated host-pure
(`software/tests/test_cal_host.py`) against an analytic cavity instead.

Step-for-step vs that session:

| reference cell (as invoked) | here |
|---|---|
| `Resonator`: q0 wideband 6.53 → 6.85 GHz / 1000 pts, then q1–q7 ± 25 MHz / 250 pts, 2048 shots | same two scans, same shots — the \|0⟩ sweep is summed **on-core** (two words a point), so each is ONE run. Characterization: nothing is proposed and nothing is written, so the reference's save/restore block has no counterpart |
| `ReadoutCalibration([0..7], gate='X90', n_shots=5000)` **×3**, then a manual `demod/phase += π/2 − φ` cell | **one** run, 1000 shots. Our proposal is ABSOLUTE — the axis is measured in the zero demod frame, so re-running proposes the same phase (a fixed point) and the reference's repeat-until-converged loop is unnecessary. Axis → **+real** (where the on-chip `sign(sumR)` cuts), not the reference's +imag (spec 13 Q2). The manual cell is reproduced right after the run, translated to those two conventions |
| `Separation`: `readout/{q}/freq` ± 2.5 MHz, 31 pts, 3000 shots | same span/points; argmax of the two-state **cluster SNR** (qcal's own formula); RAW shots RAM-sized (spec 13 §5) |
| `Separation`: `demod/time` ± 50 ns, 21 pts | `Window(knob='demod/dur')` — scored by the **confusion diagonal**, our scoring for every timing knob |
| `Separation`: `readout/{q}/time` ± 150 ns, 21 pts | `Window(knob='dur')` |
| `Separation`: `demod/delay` ± 100 ns, 31 pts **and** `Fidelity`: `demod/delay` ± 100 ns, 21 pts | **one** `Window(knob='demod/delay')` cell — our scoring is already the confusion diagonal the second of those uses |
| `Separation`: demod `ramp_fraction` ± 0.1 | — not run: there is no envelope-kwargs knob and `save_qcal` does not persist kwargs; the reference cell also has a sign bug (`− cfg[...]`) and no saved output (spec 20 §8) |
| `Fidelity`: `readout/{q}/amp` ± 0.005, 31 pts, 3000 shots | same |
| `ReadoutFidelity`, 5000 shots, fixed classifier | same — heralded confusion matrix |
| `Frequency([0..7], detunings=±1, ±2 MHz, t_max=1 µs, 512 shots)` | same — V-fit `a·|x−b|+c`, all 8 qubits in one run |
| `Amplitude(X90)`: coarse 0 → 1.0 / 31 pts, then fine 0.7–1.3× / 31 pts, `n_gates=4` | same two passes |
| `Amplitude(X)`: coarse 0 → 1.0 / 100 pts, then fine 0.8–1.2× / 51 pts, `n_gates=4` | same two passes at **31 points each** — points are run time, not protocol |
| `Phase(X90)`: **one absolute pass**, all 8, ± 0.3 / 31 pts | same (qcal's `relative_phase` defaults False and this session never sets it) |
| `Phase(X)`: ± 0.3 / 101 pts | same at 31 points. The reference writes `X/pulse/1/kwargs/phase`, but X6Y3's X drive sits at pulse index **0** (spec 13 Q0) — ours moves `qubit/{q}/x/phase`, the axis of the pulse that actually plays |
| FAST-DRAG, RPE, LinearResponse B-matrices, CZ | out of scope here — [`calibration_process_x6y3.ipynb`](calibration_process_x6y3.ipynb) covers all but `LinearResponse` (spec 20 §8) |

Deliberate differences (spec 13 §2): sweeps are **on-core loops**, discrimination is the **on-chip
`res` bit**, and a failed fit **refuses** to update the config — per qubit, so one drifted qubit
does not veto the other seven. Every cell reloads the working tree (the reference's `cfg.load()`);
applied proposals are saved back — the full qcal round trip.

In [ ]:
import shutil
from pathlib import Path

import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d   # qcal's analyze: smooth, then hunt the notch

import riscq
from riscq.cal import (Config, Resonator, ReadoutCalibration, Separation, Window, Fidelity,
                       ReadoutFidelity, Frequency, Amplitude, Phase)
from riscq.driver.remote import RemoteDriver, upload_bundle
from riscq.map import SocMap, SocParams
from riscq.pulses import units

MHz, us, ns = 1e6, 1e-6, 1e-9

## Connect to the board and the config of record

The reference preamble points at a chassis (`ip_address`/`port`) and a config dir. Here the chassis
is the ZCU216 board server ([docs/software/board-server.md](../docs/software/board-server.md)) with
an X6Y3 gateware bundle loaded, and the config is the real qcal tree, copied to a **working file**
that every cell reloads and every applied proposal is saved back into.

In [ ]:
BOARD = '192.168.1.122'                   # the ZCU216's LAN address (or the full PYRO: uri)
PORT = 9091

SW = Path(riscq.__file__).resolve().parents[1]              # .../software
SRC = SW.parent / 'examples' / 'cal-config-x6y3.yaml'       # the real X6Y3 qcal tree
WORK = SW / 'build' / 'x6y3_config.yaml'                    # the working copy (the reference's basedir)
WORK.parent.mkdir(exist_ok=True)
shutil.copy(SRC, WORK)

drv = RemoteDriver(BOARD, PORT)
print('server:', drv.board.info())

# first time only — push the X6Y3 build up and load it:
# upload_bundle(drv, 'x6y3', xsa='../build/x6y3/top.xsa',
#               params_json='../software/configs/x6y3.json')
# info = drv.board.load('x6y3')
# assert info['mts_result'] == 0, 'multi-tile sync missed its target latencies'

m = SocMap(SocParams.from_json(drv.board.get_params()))
QUBITS = list(range(8))

cfg = Config.from_qcal(WORK)
cfg.check_hardware(m.params)     # the tree's DAC/ADC rates + interpolation must describe THIS bundle
print(f"connected to '{m.params.name}': {m.params.qubit_num} cores, "
      f"{units.sample_rate(m.params) / 1e9:.0f} GS/s DACs;  herald = {cfg['readout/herald']}")

def step(cal):
    '''run -> print -> apply the qubits that passed -> persist (per-qubit write-back, spec 13 §2).'''
    r = cal.run(drv)
    bad = sorted(q for q, v in r.oks.items() if not v)
    print(f'{r.label}: ok={r.ok}  proposal={r.proposal}')
    if r.ok or any(r.oks.values()):
        r.apply()                                # writes only the passing qubits' paths
        cfg.save_qcal(WORK)
        if bad:
            print(f'  qubits {bad} failed — their config left unchanged (recentre/widen and re-run)')
    else:
        print('  fit failed — config left unchanged (fail-loud, spec 13 §2)')
    return r

# Resonator Spectroscopy

Reference: two `Resonator` scans with every qubit left in |0⟩ — a **wideband** one on q0
(6.53 → 6.85 GHz, 1000 points, 2048 shots) that walks the whole readout band, then a **± 25 MHz**
scan on q1–q7 (250 points) around each resonator's stored frequency. Characterization, not
calibration: the session saves the params it sweeps under (`readout/{q}/{freq, time, amp}`,
`demod/time`, `readout/herald`, `reset/passive/delay`) and puts them back afterwards.

`Resonator` is the |0⟩ leg of `Separation`'s sweep with the prep and the discrimination taken away —
the measurement you can run before any of the readout is calibrated. What it adds is the WIDTH:
k_vna in **IQSUM** mode sums each point's shots **on-core**, and coherently (qcal's own
`iq.mean(axis=1)` — the reason its figure has a phase panel), so a point costs two words instead of
2·shots and the reference's 1000-point scan is ONE run inside the core's 16 KB. 1024 points is the
ceiling; wider scans split, as [`vna.ipynb`](vna.ipynb) does across the full Nyquist zone.

Three translations:

- **nothing is written, so nothing is restored.** The class proposes nothing (like `Punchout`):
  `data[q]` is the mean IQ, its magnitude and the realized frequency axis, and the cell picks the
  extremum — qcal reads a **dip** (`analyze` smooths the dB curve and runs `find_peaks` on its
  negation, falling back to `argmin`), the notch geometry of a hanger-coupled resonator. These are
  the only two cells here that don't go through `step()`, and the reference's save/restore block has
  no counterpart.
- **the idle head is shortened on the loaded copy.** The sweep never excites a qubit, so what it
  waits for is resonator ring-down, not the tree's 500 µs T1 relax — each cell drops `reset/relax`
  on the tree it loaded, and since neither cell calls `save_qcal`, that *is* the reference's restore.
- **the reference's own 2048 shots.** They cost run time only (the sum is on-core, never in RAM),
  and dropping the relax head is what keeps them affordable: X6Y3's grid comes out at 3.3 µs a shot,
  so the wideband scan fires for ~7 s and the ± 25 MHz one for ~2 s. At the tree's 500 µs head the
  same shots would be 17 minutes.

The scans locate the resonators; `Separation` below is what calibrates the probe, and the two
answers differ on purpose — the |0⟩ notch is not the max-separation frequency (spec 13 §5).

In [ ]:
cfg = Config.from_qcal(WORK)
cfg['reset/relax'] = 2 * us            # |0>-only: ring-down, not T1 (never saved back)

freqs = {0: np.linspace(6.53e9, 6.85e9, 1000)}                  # the session's wideband scan
spec = Resonator(cfg, [0], freqs=freqs, shots=2048).run(drv)    # 1000 points = 8 KB of IQ sums, one run

d = spec.data[0]
db = 20 * np.log10(d['mag'])
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 6), sharex=True)   # qcal's plot(interactive=True)
ax1.plot(d['x'] / 1e9, db, lw=0.8); ax1.set_ylabel('|S| [dB]')
ax2.plot(d['x'] / 1e9, np.unwrap(np.angle(d['iq'])), lw=0.8)
ax2.set_ylabel('unwrapped phase [rad]'); ax2.set_xlabel('readout frequency [GHz]')
for q in QUBITS:
    ax1.axvline(cfg[f'readout/{q}/freq'] / 1e9, color='gray', ls='--', lw=0.6)   # the tree's 8 probes
plt.show()
print(f"deepest notch: {d['x'][np.argmin(gaussian_filter1d(db, 3))] / 1e9:.6f} GHz")   # qcal's rule

In [ ]:
cfg = Config.from_qcal(WORK)
cfg['reset/relax'] = 2 * us
qubits = [1, 2, 3, 4, 5, 6, 7]                  # q0 is the wideband scan above (the reference's split)
freqs = {q: np.linspace(-25, 25, 250) * MHz + cfg[f'readout/{q}/freq'] for q in qubits}
spec = Resonator(cfg, qubits, freqs=freqs, shots=2048).run(drv)    # all seven swept simultaneously

for q in qubits:
    d = spec.data[q]
    plt.plot((d['x'] - cfg[f'readout/{q}/freq']) / MHz, 20 * np.log10(d['mag']), label=f'q{q}')
plt.xlabel('readout freq − stored [MHz]'); plt.ylabel('|S| [dB]'); plt.legend(ncol=4); plt.show()
for q in qubits:
    d = spec.data[q]
    notch = float(d['x'][np.argmin(gaussian_filter1d(20 * np.log10(d['mag']), 3))])
    print(f"q{q}: notch {notch / 1e9:.6f} GHz  "
          f"({(notch - cfg[f'readout/{q}/freq']) / 1e6:+.3f} MHz from the tree — not written back)")

# Readout Calibration

Reference: `ReadoutCalibration(..., gate='X90', n_shots=5000, n_levels=2)` — prep |0⟩ and |1⟩
(X90·X90) on all 8 qubits simultaneously, cluster the raw IQ, persist the classifier. The proposal
fixes each qubit's `demod/phase` so the |0⟩→|1⟩ cluster axis lands on +real, where the on-chip
`sign(sumR)` discriminates (spec 13 Q2). The host-side clustering is the reference's own scheme
(spec 21): an **unsupervised GMM on the pooled shots** — prep labels never place the boundary, so
state-prep errors do not bias it — and `plot()` is qcal's figure, the shot density under the
fitted decision regions. Raw-IQ shots are captured in one rerun per prep state, so they are
core-RAM-bounded: 1000 shots (8 KB of 16 KB) instead of the reference's host-streamed 5000.

In [ ]:
cfg = Config.from_qcal(WORK)
cal = ReadoutCalibration(cfg, QUBITS, shots=1000, gate='X90')
rc = step(cal)

print('separation:', {q: round(rc.data[q]['separation'], 3) for q in QUBITS})
cal.plot()             # qcal's readout figure: shot density + the GMM decision regions
plt.show()

### The reference's manual demod-phase cell

The session follows its `ReadoutCalibration` runs with a hand-written update: φ = the angle of the
raw |0⟩→|1⟩ prep-mean difference, then `demod/phase += π/2 − φ` and `cfg.save()`. Reproduced here
with two convention translations (spec 13 Q2), after which it is **algebraically the phase
`ReadoutCalibration` already proposed** (`π − φ` wrapped ≡ `−angle(m0 − m1)`, readout.py's formula)
— so this cell is a cross-check against the value `step()` above already applied, differing only by
raw-prep-means vs fitted-GMM-means (shot noise when the clusters are clean):

- **incremental → absolute**: qcal captures IQ in the *current* demod frame, so its correction is a
  nudge and the session repeats the run/update pair until it converges. Our capture runs in the
  **zero** demod frame (`_rawiq_prog`), so the same measurement sets the phase outright — assign,
  don't `+=` — and re-running proposes the same phase again (the fixed point the reference iterates
  toward).
- **target +imag → target +real**: qcal parks the |0⟩→|1⟩ axis on +imag, where its software
  threshold cuts. The on-chip discriminator is `sign(sumR)` — a hard zero on the **real** axis with
  |0⟩ on the + side — so the target angle is π, not π/2.

In [ ]:
cfg = Config.from_qcal(WORK)
qubits = QUBITS                            # the reference cell ran qubits = [0]
for q in qubits:
    m0 = np.mean(rc.data[q]['iq0'] @ [1, 1j])   # raw prep-state means as complex (the reference's
    m1 = np.mean(rc.data[q]['iq1'] @ [1, 1j])   #   np.mean over cal._circuits' iq_data)
    phi = np.arctan2((m1 - m0).imag, (m1 - m0).real)
    # reference: cfg[f'readout/{q}/demod/phase'] += np.pi/2 - phi
    phase = float(np.angle(np.exp(1j * (np.pi - phi))))       # π − φ, wrapped to (−π, π]
    print(f"q{q}: φ={phi:+.4f}  manual={phase:+.4f}  "
          f"proposal={rc.proposal[f'readout/{q}/demod/phase']:+.4f} rad")
    cfg[f'readout/{q}/demod/phase'] = phase
cfg.save_qcal(WORK)                        # the reference's cfg.save()

## Separation

Reference: sweep `readout/{q}/freq` ± 2.5 MHz (31 points) around each qubit's current value, both
prep states, argmax of the cluster SNR (`‖Δmeans‖ / Σ(2√cov)`). The matched DAC+demod pair is swept
on-core in RAW mode; shots beyond the 16 KB-RAM budget (33/point/prep at 31 points) split into
extra reruns of the one resident image (spec 13 §5). A qubit whose argmax lands on the sweep edge
is flagged not-ok (session drift to or past the span — widen and re-run; X6Y3's hybridised q4/q5
pair hops MHz between sessions). After the freqs move, re-run ReadoutCalibration: the demod phase
depends on the detuning.

In [ ]:
cfg = Config.from_qcal(WORK)
# 256 shots/point/prep = 8 rerun pairs, ~10 s at the 500 µs grid; widen span= after a long gap
sep = step(Separation(cfg, QUBITS, span=2.5 * MHz, points=31, shots=256, gate='X90'))

for q in QUBITS:
    plt.plot((sep.data[q]['x'] - sep.data[q]['x'].mean()) / MHz, sep.data[q]['y'], '-', label=f'q{q}')
plt.xlabel('readout freq − centre [MHz]'); plt.ylabel('cluster SNR'); plt.legend(ncol=4); plt.show()
for q in QUBITS:
    print(f"readout/{q}/freq -> {cfg[f'readout/{q}/freq'] / 1e9:.6f} GHz")

## Readout timing — the three window knobs

Reference: three more `Separation` sweeps (`demod/time` ± 50 ns / 21, `readout/{q}/time`
± 150 ns / 21, `demod/delay` ± 100 ns / 31) plus a `Fidelity` sweep of `demod/delay` (± 100 ns / 21),
each centred **per qubit** on that qubit's current value.

Here all three are `Window`, the one class that moves a readout timing, with `knob` naming the path
it writes; `durs` takes a `{q: [seconds]}` dict so each qubit sweeps around **its own** timing while
all of them are still read out simultaneously (spec 13 §8, 20 U3). One cell covers the reference's
two `demod/delay` sweeps: `Window` already scores the **confusion diagonal** under the fixed on-chip
discriminator, which is the `Fidelity` version's statistic.

Two documented deviations (spec 20 §2): a duration sweep plays a **truncated** `cosine_square` — the
program is compiled at the longest candidate and only the slot's `dur` is retuned per point, where
qcal reshapes the ramp per point (a second-order apodization difference over these ± 50–150 ns
spans); and an argmax on the sweep edge is accepted here, unlike `Separation`'s peak — a confusion
diagonal is legitimately monotone up to saturation, so an edge winner is not evidence of drift.

In [ ]:
for knob, span in (('demod/dur', 50 * ns),      # ref: Separation on demod/time
                   ('dur', 150 * ns),           # ref: Separation on the readout DRIVE length
                   ('demod/delay', 100 * ns)):  # ref: Separation + Fidelity on the ADC round trip
    cfg = Config.from_qcal(WORK)                # the reference's cfg.load() per sweep
    durs = {q: np.linspace(-span, span, 21) + cfg[f'readout/{q}/{knob}'] for q in QUBITS}
    w = step(Window(cfg, QUBITS, durs=durs, shots=2000, gate='X90', knob=knob))

    for q in QUBITS:
        plt.plot((durs[q] - durs[q].mean()) / ns, w.data[q]['y'], '-', label=f'q{q}')
    plt.xlabel(f'{knob} − centre [ns]'); plt.ylabel('confusion diagonal')
    plt.title(f'{knob}  (± {span / ns:.0f} ns, 21 pts)'); plt.legend(ncol=4); plt.show()
    for q in QUBITS:
        print(f"readout/{q}/{knob} -> {cfg[f'readout/{q}/{knob}'] / ns:.1f} ns")

## Fidelity

Reference: sweep `readout/{q}/amp` ± 0.005 (31 points, 3000 shots) under the **fixed** classifier,
argmax of the confusion diagonal. Same knob and scoring — the discriminator is the `res` bit whose
phase ReadoutCalibration fixed, never retrained per point. (The reference disables ESP here; ESP is
a 3-level non-goal, spec 13 §11 — this chain is 2-level throughout. X6Y3's smallest readout amp is
0.0115, so the ± 0.005 span reaches down to 0.0065 — swept in full, no floor.)

In [ ]:
cfg = Config.from_qcal(WORK)
fid = step(Fidelity(cfg, QUBITS, amp_span=0.005, points=31, shots=3000, gate='X90'))

for q in QUBITS:
    plt.plot(fid.data[q]['x'], fid.data[q]['y'], '-', label=f'q{q}')
plt.xlabel('readout amp'); plt.ylabel('confusion diagonal'); plt.legend(ncol=4); plt.show()
for q in QUBITS:
    print(f"readout/{q}/amp -> {cfg[f'readout/{q}/amp']:.5f}")

# Readout Fidelity

Reference: `ReadoutFidelity(..., classifier=classifier, n_shots=5000)` — the confusion matrix under
the classifier passed in, no retraining. Same here: two COUNTS reruns at the calibrated amp,
straight off the `res` bit, heralded (every counts-mode shot on this `herald: true` config is
post-selected on a pre-sequence |0⟩ read, exactly like qcal's transpiler).

In [ ]:
cfg = Config.from_qcal(WORK)
rof = step(ReadoutFidelity(cfg, QUBITS, shots=5000, gate='X90'))

for q in QUBITS:
    print(f"q{q} confusion (row = prepared, col = classified):")
    print(np.round(rof.data[q]['confusion'], 3), f"  fidelity={rof.data[q]['fidelity']:.3f}")

# Single Qubit
## GE
### Freq

Reference: Ramsey vs detuning on **all 8 qubits** — `detunings = [-2, -1, 1, 2] MHz` (qcal's own
default set), `t_max = 1 µs`, 512 shots; fit `a·|x − b| + c` over the unsigned fringe frequencies
and correct each carrier by its vertex.

This is the cell where the per-qubit verdict earns its keep (spec 20 U0): eight qubits in one run,
and `apply()` writes the ones whose V-fit landed — a qubit whose true detuning has drifted outside
`± 2 MHz` refuses on its own without vetoing the other seven.

In [ ]:
cfg = Config.from_qcal(WORK)
detunings = np.array([-2, -1, 1, 2]) * MHz          # the session's set (qcal's default)
fr = step(Frequency(cfg, QUBITS, detunings=detunings, t_max=1 * us, points=30, shots=512))

for q in QUBITS:
    d = fr.data[q]
    plt.plot(d['applied'], d['obs'], 'o-', label=f'q{q}')
plt.xlabel('applied detuning [codes]'); plt.ylabel('|fringe| [codes]'); plt.legend(ncol=4); plt.show()
for q in QUBITS:
    print(f"qubit/{q}/freq -> {cfg[f'qubit/{q}/freq'] / 1e9:.6f} GHz")

### Amplitude X90

Reference (all 8 qubits): a coarse `n_gates=1` Rabi over the **full** amplitude range
(`np.linspace(0, 1.0, 31)`), then the fine pass — 4 repeated X90s with `relative_amp=True` sweeping
0.7–1.3× the coarse result (each X90 is a quarter period, so the train must be a multiple of 4 —
qcal's own guard). The coarse fit recovers the Rabi RATE and the amplitude that turns by exactly
π/2; the fine pass amplifies the residual error into a parabola vertex.

In [ ]:
cfg = Config.from_qcal(WORK)
ac = step(Amplitude(cfg, QUBITS, gate='X90', n_gates=1, amp_span=(0.0, 1.0), points=31, shots=512))

for q in QUBITS:
    plt.plot(ac.data[q]['x'] / units.AMP_SCALE, ac.data[q]['y'], '-', label=f'q{q}')
plt.xlabel('X90 amp'); plt.ylabel('P(|1>)'); plt.legend(ncol=4); plt.show()
for q in QUBITS:
    print(f"q{q}: coarse X90 amp -> {cfg[f'qubit/{q}/x90/amp']:.5f}")

In [ ]:
cfg = Config.from_qcal(WORK)
af = step(Amplitude(cfg, QUBITS, gate='X90', n_gates=4, amp_span=(0.7, 1.3), relative_amp=True,
                    points=31, shots=512))
for q in QUBITS:
    print(f"q{q}: fine X90 amp -> {cfg[f'qubit/{q}/x90/amp']:.5f}")

### Amplitude X180

Reference: the same two passes on the **X** — coarse `np.linspace(0, 1.0, 100)` at `n_gates=1`, then
`np.linspace(0.8, 1.2, 51)` at `n_gates=4` with `relative_amp=True`. Run here at 31 points each
(point count is run time, not protocol).

X6Y3's X is a **real pulse**, not a double-amplitude X90: a FAST_DRAG of twice the duration at a
similar amplitude (`qubit/{q}/x/*`, spec 13 Q0), so it has to be calibrated on its own. `gate='X'`
picks that pulse, targets a π turn (qcal's `period_frac` 0.5 against the X90's 0.25) and writes
`qubit/{q}/x/amp`; the repetition guard relaxes to a multiple of **2**, since 2·X = 2π.

In [ ]:
cfg = Config.from_qcal(WORK)
xc = step(Amplitude(cfg, QUBITS, gate='X', n_gates=1, amp_span=(0.0, 1.0), points=31, shots=512))

for q in QUBITS:
    plt.plot(xc.data[q]['x'] / units.AMP_SCALE, xc.data[q]['y'], '-', label=f'q{q}')
plt.xlabel('X amp'); plt.ylabel('P(|1>)'); plt.legend(ncol=4); plt.show()
for q in QUBITS:
    print(f"q{q}: coarse X amp -> {cfg[f'qubit/{q}/x/amp']:.5f}")

In [ ]:
cfg = Config.from_qcal(WORK)
xf = step(Amplitude(cfg, QUBITS, gate='X', n_gates=4, amp_span=(0.8, 1.2), relative_amp=True,
                    points=31, shots=512))
for q in QUBITS:
    print(f"q{q}: fine X amp -> {cfg[f'qubit/{q}/x/amp']:.5f}")

### Phase X90

Reference: **one absolute pass** over all 8 qubits — `phases = np.linspace(-0.3, 0.3, 31)`, with
qcal's `relative_phase` left at its default `False` (this session never sets it). The swept knob is
the X90's **virtual-Z pair** (`qubit/{q}/x90/vz` — nonzero and asymmetric on this config, e.g. q6);
each point runs qcal's two sequences (`Y180_X90` / `X180_Y90`) and the calibrated value is the
crossing of the two fitted lines (spec 13 Q3), written to **both** slots as qcal does — which is
why the calibration collapses an asymmetric stored pair into a symmetric one.

In [ ]:
cfg = Config.from_qcal(WORK)
ph = step(Phase(cfg, QUBITS, span=0.3, points=31, shots=512))   # one absolute pass, all 8

for q in QUBITS:
    d = ph.data[q]
    plt.plot(d['x'], d['p0'], '-', label=f'q{q} Y180_X90')
    plt.plot(d['x'], d['p1'], '--', label=f'q{q} X180_Y90')
plt.xlabel('swept virtual-Z [rad]'); plt.ylabel('P(|1>)'); plt.legend(ncol=4, fontsize=6); plt.show()
for q in QUBITS:
    vz = cfg.get(f'qubit/{q}/x90/vz', [0.0, 0.0])
    print(f'q{q}: X90 virtual-Z pair -> [{vz[0]:+.4f}, {vz[1]:+.4f}] rad')

### Phase X180

Reference: `Phase(gate='X', phases=np.linspace(-0.3, 0.3, 101))`, all 8 — run here at 31 points.

This is a different knob from the X90's: the X carries no virtual-Z pair, so what is calibrated is
the pulse's **own axis phase**. The circuit is one `X90 · X · X90` composite — a 2π rotation that
only returns to |0⟩ when the X sits on the X90s' axis — so P(|1⟩) is cosinusoidal in the swept axis
and the calibrated value is its **minimum** (qcal fits the same cosine to P(|0⟩) and takes the
maximum). The fringe runs at twice the swept axis (a π rotation's axis enters as 2φ), so the two
solutions a period apart are the same gate; the fit takes the one nearest the sweep centre, which
keeps an already-calibrated qubit from jumping.

**Reference-side caveat** (spec 20 §1): qcal hardcodes `single_qubit/{q}/GE/X/pulse/1/kwargs/phase`,
but X6Y3's X drive is at pulse index **0** — the reference cell writes a path that is not this
chip's X. Ours writes `qubit/{q}/x/phase`, which `save_qcal` maps back onto the drive pulse
whatever its index.

In [ ]:
cfg = Config.from_qcal(WORK)
px = step(Phase(cfg, QUBITS, gate='X', span=0.3, points=31, shots=512))

for q in QUBITS:
    plt.plot(px.data[q]['x'], px.data[q]['y'], '-', label=f'q{q}')
plt.xlabel('X axis phase [rad]'); plt.ylabel('P(|1>)'); plt.legend(ncol=4); plt.show()
for q in QUBITS:
    print(f"q{q}: X axis phase -> {cfg.get(f'qubit/{q}/x/phase', 0.0):+.4f} rad")

## Summary — the calibrated tree

Every step ran through the full qcal round trip: `Config.from_qcal` → calibrate → `save_qcal` back
into the working tree. Diff `WORK` against `cal-config-x6y3.yaml` to see exactly what moved.

In [ ]:
cfg = Config.from_qcal(WORK)
for q in QUBITS:
    vz = cfg.get(f'qubit/{q}/x90/vz', [0.0, 0.0])
    print(f"q{q}: f_ge={cfg[f'qubit/{q}/freq'] / 1e9:.6f} GHz  "
          f"x90 amp={cfg[f'qubit/{q}/x90/amp']:.5f} vz=[{vz[0]:+.4f},{vz[1]:+.4f}]  "
          f"x amp={cfg[f'qubit/{q}/x/amp']:.5f} phase={cfg.get(f'qubit/{q}/x/phase', 0.0):+.4f}")
    print(f"     readout {cfg[f'readout/{q}/freq'] / 1e9:.6f} GHz @ {cfg[f'readout/{q}/amp']:.5f}  "
          f"drive={cfg[f'readout/{q}/dur'] / ns:.0f} ns "
          f"window={cfg[f'readout/{q}/demod/dur'] / ns:.0f} ns "
          f"delay={cfg[f'readout/{q}/demod/delay'] / ns:.0f} ns  "
          f"fidelity={rof.data[q]['fidelity']:.3f}")
print(f'\ncalibrated tree: {WORK}')

## Disconnect

In [ ]:
drv.close()
print('disconnected')